In [0]:
employees_data = [(1, "Alice", 10),
                   (2, "Ben", 20),
                   (3, "Cara", 10),
                   (4, "Dev", 99)]   # 99 doesn't exist in departments - orphan on purpose
employees = spark.createDataFrame(employees_data, ["id", "name", "department_id"])

departments_data = [(10, "IT"), (20, "HR"), (30, "Finance")]
departments = spark.createDataFrame(departments_data, ["department_id", "department_name"])

employees.display()
departments.display()

In [0]:
# Inner Join — Only Matching Rows

#Only rows where department_id exists in both DataFrames are kept. If an employee has a department_id that doesn't exist in the departments table, that employee is dropped entirely from the result.

#Note: inner is actually the default — employees.join(departments, on="department_id") without specifying how does an inner join. It's still worth writing how="inner" explicitly in real code, so anyone reading it doesn't have to know the default by heart

employees.join(departments, on="department_id", how="inner").display()



In [0]:
#Left Join — Keep Everything from the Left Side
#Every row from employees is kept, whether or not it has a matching department. Rows with no match get null values in all the departments columns.

#Note ⚠️ This is the join type you'll use constantly for data quality work: “keep every row from my main table, and show me which ones don't have a match.” The rows that come back with nulls in the joined columns ARE the data quality issue you're looking for — orphaned records, broken foreign keys, missing reference data

employees.join(departments, on="department_id", how="left").display()   

In [0]:
# Right Join and Full Outer Join

#In practice, right joins are rare — most people just swap the DataFrame order and use left instead, since it reads more naturally. full/outer joins are more common in reconciliation work (Day 39), where you need to see records missing from either side.

employees.join(departments, on="department_id", how="right").display()
employees.join(departments, on="department_id", how="full").display()

In [0]:
#Joining on Differently Named Columns

#When the key isn't named the same in both DataFrames, on="column_name" won't work — you need an explicit condition


employees.join(departments, employees.department_id == departments.department_id, how="left").display()


In [0]:
# Finding Orphaned Records - the Core DQ Pattern for joins

orphans = employees.join(departments, on="department_id",how="left").filter(departments.department_id.isNull())
orphans.display()

In [0]:
# Putting it all together - Example
# Tables: orders (order_id, customer_id, amount) and customers (customer_id, name, region)

orders_data = [(101, 1, 250.00),
               (102, 2, 150.00),
               (103, 1, 300.00),
               (104, 3, 175.00),
               (105, 99, 500.00)]  # customer_id 99 doesn't exist - orphan
orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "amount"])

customers_data = [(1, "Alice", "East"),
                  (2, "Bob", "West"),
                  (3, "Charlie", "East"),
                  (4, "Diana", "South")]  # customer_id 4 has no orders
customers = spark.createDataFrame(customers_data, ["customer_id", "name", "region"])

orders.display()
customers.display()

In [0]:
from pyspark.sql.functions import col
# Enrich orders with customer info, keeping every order even if the customer is missing
encriched = orders.join(customers, on="customer_id", how="left")
encriched.display()
# Find customers with no orders

orphaned_order = encriched.filter(col("name").isNull())

print("Total orders:", orders.count())
print("Orphaned orders:", orphaned_order.count())
orphaned_order.display()



